# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1_sJaD57gHuiz1joEgl4B-u0aDy8jtMDo)



## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


In [1]:
import numpy as np

class WarehouseMDP:
    def __init__(self):
        self.height = 5
        self.width = 6

        # Estado inicial
        self.start = (0, 0)

        # Obstáculos / estanterías
        self.walls = {(0, 3), (1, 1), (2, 4), (4, 2)}

        # Celdas de piso resbaloso
        self.slippery_states = {(1, 2), (2, 1), (3, 3)}

        # Estados terminales y sus recompensas
        self.terminal_states = {
            (0, 5): 10.0,
            (2, 2): 2.0,
            (3, 5): -10.0,
        }

        # Peligros no terminales
        self.danger_states = {
            (1, 4): -3.0,
            (4, 1): -3.0,
        }

        self.living_reward = -1.0
        self.gamma = 0.9

        # Acciones: UP, DOWN, LEFT, RIGHT
        self.actions = [
            (-1, 0),  # UP
            ( 1, 0),  # DOWN
            ( 0,-1),  # LEFT
            ( 0, 1),  # RIGHT
        ]

    def is_valid_state(self, state):
        row, col = state
        if not (0 <= row < self.height and 0 <= col < self.width):
            return False
        return state not in self.walls

    def states(self):
        return [
            (row, col)
            for row in range(self.height)
            for col in range(self.width)
            if self.is_valid_state((row, col))
        ]

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        # R(s) según la convención del notebook
        if self.is_terminal(state):
            return self.terminal_states[state]
        if state in self.danger_states:
            return self.danger_states[state]
        return self.living_reward

    def get_transition_probs(self, state, action):
        """
        Devuelve:
            [(next_state, probability), ...]

        Recuerda:
        - las probabilidades dependen de si 'state' es resbaloso;
        - si golpea pared/borde, next_state = state.
        """
        if self.is_terminal(state):
            return [(state, 1.0)]

        if state in self.slippery_states:
            p_intended = 0.60
            p_perp = 0.20
        else:
            p_intended = 0.90
            p_perp = 0.05

        perp1 = (action[1], action[0])
        perp2 = (-action[1], -action[0])

        outcomes = [
            (action, p_intended),
            (perp1, p_perp),
            (perp2, p_perp),
        ]

        transitions = []
        for next_action, prob in outcomes:
            next_state = (
                state[0] + next_action[0],
                state[1] + next_action[1],
            )

            if not self.is_valid_state(next_state):
                next_state = state

            transitions.append((next_state, prob))

        return transitions



### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [2]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")


Número de estados: 26
✓ Todas las distribuciones de transición suman 1.



## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [3]:
def expected_next_value(grid, state, action, V):
    # sum_{s'} T(s,a,s') V(s')
    return sum(
        prob * V[next_state]
        for next_state, prob in grid.get_transition_probs(state, action)
    )


def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    V = {state: 0.0 for state in grid.states()}

    for iteration in range(max_iter):
        V_new = V.copy()
        biggest_change = 0.0

        for state in grid.states():
            if grid.is_terminal(state):
                V_new[state] = grid.get_reward(state)
            else:
                best_expected_value = max(
                    expected_next_value(grid, state, action, V)
                    for action in grid.actions
                )
                V_new[state] = (
                    grid.get_reward(state)
                    + grid.gamma * best_expected_value
                )

            biggest_change = max(
                biggest_change,
                abs(V_new[state] - V[state])
            )

        V = V_new

        if biggest_change < threshold:
            break

    return V, iteration + 1


def extract_policy(grid, V):
    policy = {}

    for state in grid.states():
        if grid.is_terminal(state):
            continue

        policy[state] = max(
            grid.actions,
            key=lambda action: expected_next_value(grid, state, action, V)
        )

    return policy



## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$

In [4]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    V = {state: 0.0 for state in grid.states()}

    for iteration in range(max_iter):
        V_new = V.copy()
        biggest_change = 0.0

        for state in grid.states():
            if grid.is_terminal(state):
                V_new[state] = grid.get_reward(state)
            else:
                action = policy[state]
                V_new[state] = (
                    grid.get_reward(state)
                    + grid.gamma * expected_next_value(grid, state, action, V)
                )

            biggest_change = max(
                biggest_change,
                abs(V_new[state] - V[state])
            )

        V = V_new

        if biggest_change < threshold:
            break

    return V, iteration + 1


def policy_improvement(grid, V):
    new_policy = {}

    for state in grid.states():
        if grid.is_terminal(state):
            continue

        new_policy[state] = max(
            grid.actions,
            key=lambda action: expected_next_value(grid, state, action, V)
        )

    return new_policy


def policy_iteration(grid, threshold=1e-4, max_iter=100):
    # Política inicial arbitraria: RIGHT en todos los estados no terminales
    initial_action = (0, 1)
    policy = {
        state: initial_action
        for state in grid.states()
        if not grid.is_terminal(state)
    }

    history = []

    for iteration in range(max_iter):
        V, eval_iterations = policy_evaluation(grid, policy, threshold=threshold)
        new_policy = policy_improvement(grid, V)

        changed = sum(
            new_policy[state] != policy[state]
            for state in new_policy
        )

        history.append({
            "policy_iteration": iteration + 1,
            "evaluation_sweeps": eval_iterations,
            "changed_actions": changed,
        })

        policy = new_policy

        if changed == 0:
            break

    V, _ = policy_evaluation(grid, policy, threshold=threshold)

    return policy, V, history



## Parte 4 — Visualización y comparación


In [5]:
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [6]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n✓ Ambos algoritmos encontraron la misma política óptima.")


=== VALUE ITERATION ===
Iteraciones: 20

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

=== POLICY ITERATION ===
Historia: [{'policy_iteration': 1, 'evaluation_sweeps': 73, 'changed_actions': 19}, {'policy_iteration': 2, 'evaluation_sweeps': 61, 'changed_actions': 7}, {'policy_iteration': 3, 'evaluation_sweeps': 19, 'changed_actions': 1}, {'policy_iteration': 4, 'evaluation_sweeps': 22, 'changed_actions': 0}]

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +


## Parte 5 — Interpreta la política

Antes de cambiar parámetros, responde:

1. Desde `START`, ¿el robot busca la **entrega +10** o prefiere la **estación de carga +2**?
2. ¿Por qué una recompensa menor podría ser óptima?
3. ¿En qué estados el piso resbaloso cambia la decisión?
4. ¿Qué papel cumple el costo por paso `-1`?
5. ¿Por qué \(T(s,a,s')\) ya no puede implementarse con las mismas probabilidades para todos los estados?

### Experimento A — Menos costo por paso

Cambia:

```python
living_reward = -0.1
```

Predice la política **antes de ejecutar**.

### Experimento B — Piso muy resbaloso

Cambia la probabilidad de movimiento deseado del piso resbaloso:

```python
0.60 → 0.40
```

y reparte el restante entre las dos desviaciones.

### Experimento C — Más paciencia

Cambia:

```python
gamma = 0.99
```

¿La política valora más la recompensa `+10` distante?

### Bonus

Encuentra aproximadamente el valor de `living_reward` a partir del cual la política desde `START` cambia entre:

- ir a carga `+2`;
- intentar llegar a entrega `+10`.


### Respuestas a las preguntas de interpretación

#### 1. Desde `START`, ¿el robot busca la entrega +10 o prefiere la estación de carga +2?
Prefiere la **estación de carga +2** en `(2, 2)`.  
Siguiendo la política óptima $\pi^*$, la trayectoria determinista desde `START = (0, 0)` es:
$$ (0, 0) \xrightarrow{\rightarrow} (0, 1) \xrightarrow{\rightarrow} (0, 2) \xrightarrow{\downarrow} (1, 2) \xrightarrow{\downarrow} (2, 2) \quad [\text{CARGA } +2] $$
El valor óptimo en el estado inicial es $V^*(0, 0) \approx -2.575$.

#### 2. ¿Por qué una recompensa menor (+2) resulta óptima frente a una mayor (+10)?
Porque el objetivo en un MDP con descuento es maximizar el retorno acumulado esperado:
$$ G_t = \sum_{k=0}^{\infty} \gamma^k R(s_{t+k}) $$
- Llegar a la estación de carga $+2$ toma únicamente 4 transiciones.
- Llegar a la entrega $+10$ requiere al menos 7 transiciones, atravesando la celda resbalosa `(1, 2)` y la celda de peligro `(1, 4)` con recompensa $R=-3$.  
Con un costo de vida por paso de $R(s) = -1.0$ y un factor de descuento $\gamma = 0.9$, cada paso adicional acumula penalización y disminuye exponencialmente el valor presente de la meta lejana ($\gamma^7 \cdot 10 \approx 0.478 \cdot 10 = 4.78$). La suma de costos de paso y peligros a lo largo del camino largo supera la diferencia de recompensa final, haciendo que la opción más rápida y segura hacia $+2$ tenga un valor neto superior.

#### 3. ¿En qué estados el piso resbaloso cambia la decisión?
El piso resbaloso cuadruplica la probabilidad de desvío lateral ($20\%$ hacia cada lado perpendicular frente a $5\%$ en piso normal). Esto influye directamente en:
- **`(1, 2)`**: Siendo resbaloso, cualquier acción lateral tiene $20\%$ de probabilidad de empujar al robot contra la pared en `(1, 1)` o hacia arriba `(0, 2)`. La acción óptima es ir hacia abajo $\downarrow$, entrando a la seguridad de la estación de carga `(2, 2)` en lugar de arriesgarse a avanzar hacia la derecha en dirección a zonas de peligro.
- **`(2, 1)`**: Es resbaloso y se ubica directamente sobre el peligro $-3$ en `(4, 1)`. La política elige $\rightarrow$ para ingresar de inmediato al terminal seguro `(2, 2)`, evitando deambular en celdas con alta incertidumbre de movimiento.
- **`(3, 3)`**: Se ubica en la misma fila del peligro mortal $-10$ en `(3, 5)`. Al no tener control firme, la política prefiere moverse hacia arriba $\uparrow$, alejándose de la fila que conduce al peligro mortal.

#### 4. ¿Qué papel cumple el costo por paso `-1`?
Actúa como un incentivo de urgencia: penaliza al agente por cada unidad de tiempo que permanece en el almacén sin alcanzar un terminal. Un costo por paso negativo severo induce al agente a finalizar el episodio en el terminal más próximo disponible, aunque su recompensa sea menor. Si el costo por paso disminuye (acercándose a cero), la urgencia decae y el robot tiene margen para preferir trayectorias más largas hacia recompensas de mayor magnitud.

#### 5. ¿Por qué $T(s, a, s')$ ya no puede implementarse con las mismas probabilidades para todos los estados?
Porque la dinámica del entorno es espacialmente heterogénea: depende de las características físicas de la celda actual $s$:
- En piso normal: $P(\text{intencionada}) = 0.90$, $P(\text{desvío izq}) = 0.05$, $P(\text{desvío der}) = 0.05$.
- En piso resbaloso: $P(\text{intencionada}) = 0.60$, $P(\text{desvío izq}) = 0.20$, $P(\text{desvío der}) = 0.20$.
- En terminales: el estado es absorbente, $P(s \mid s, a) = 1.0$.  
Por ende, la función de transición debe consultar el estado $s$ para determinar la distribución de probabilidad aplicable.

---
### Experimento A — Menos costo por paso (`living_reward = -0.1`)

**Predicción antes de ejecutar:**  
Al reducir el costo por paso de $-1.0$ a $-0.1$, la penalización acumulada por recorrer caminos largos se vuelve prácticamente despreciable en comparación con la recompensa final de $+10$. Por tanto, el robot abandonará la ruta corta hacia la estación de carga $+2$ y la política óptima desde `START` lo guiará hacia la **entrega +10**, a pesar de requerir 7 pasos y cruzar la casilla de peligro $-3$.

In [7]:
grid_a = WarehouseMDP()
grid_a.living_reward = -0.1

V_a, n_a = value_iteration(grid_a)
pi_a = extract_policy(grid_a, V_a)

print("=== EXPERIMENTO A (living_reward = -0.1) ===")
print(f"Convergencia VI: {n_a} iteraciones")
print(f"Valor en START (0, 0): {V_a[grid_a.start]:+.3f}\n")

print("Valores V*(s):")
print_values(grid_a, V_a)

print("\nPolítica óptima π*(s):")
print_policy(grid_a, pi_a)

# Trazar la trayectoria desde START
curr = grid_a.start
path_a = [curr]
while not grid_a.is_terminal(curr):
    a = pi_a[curr]
    curr = (curr[0] + a[0], curr[1] + a[1])
    path_a.append(curr)

print("\nTrayectoria desde START:", " → ".join(str(s) for s in path_a))
print("Destino alcanzado:", "ENTREGA +10" if path_a[-1] == (0, 5) else "CARGA +2")

=== EXPERIMENTO A (living_reward = -0.1) ===
Convergencia VI: 26 iteraciones
Valor en START (0, 0): +1.649

Valores V*(s):
 +1.649 |  +1.992 |  +2.361 |   WALL   |  +8.591 | +10.000
 +1.358 |   WALL   |  +2.797 |  +3.911 |  +4.550 |  +8.591
 +1.259 |  +1.533 |  +2.000 |  +3.306 |   WALL   |  +7.537
 +1.243 |  +1.540 |  +2.040 |  +2.417 |  +2.026 | -10.000
 +0.865 |  -1.794 |   WALL   |  +2.026 |  +1.709 |  +0.874

Política óptima π*(s):
 →  |  →  |  ↓  |  #  |  →  | +10
 ↑  |  #  |  →  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  →  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

Trayectoria desde START: (0, 0) → (0, 1) → (0, 2) → (1, 2) → (1, 3) → (1, 4) → (1, 5) → (0, 5)
Destino alcanzado: ENTREGA +10


**Conclusión del Experimento A:**  
La predicción se cumple fielmente. Con `living_reward = -0.1`, el valor en `START` sube a $+1.649$ y en la casilla `(1, 2)` la acción cambia de $\downarrow$ a $\rightarrow$, alcanzando con éxito la **entrega +10** en `(0, 5)`. Con menor urgencia por paso, prima el valor terminal absoluto.

---
### Experimento B — Piso muy resbaloso (`0.60 → 0.40`)

**Predicción antes de ejecutar:**  
Al reducir la probabilidad de éxito a $0.40$, las desviaciones laterales pasan a ser del $30\%$ cada una (sumando $60\%$ de probabilidad de no avanzar hacia donde se desea). Esta alta dispersión incrementa el número esperado de pasos para salir de celdas resbalosas y eleva el riesgo de desvío hacia zonas de castigo. Por ello, el camino largo hacia la entrega resulta aún más desventajoso y arriesgado. El robot mantendrá su objetivo en la **estación de carga +2**, pero el valor esperado en `START` disminuirá (será más negativo) por efecto de la mayor incertidumbre.

In [8]:
class WarehouseMDP_Slippery(WarehouseMDP):
    def get_transition_probs(self, state, action):
        if self.is_terminal(state):
            return [(state, 1.0)]

        if state in self.slippery_states:
            p_intended = 0.40
            p_perp = 0.30
        else:
            p_intended = 0.90
            p_perp = 0.05

        perp1 = (action[1], action[0])
        perp2 = (-action[1], -action[0])

        outcomes = [
            (action, p_intended),
            (perp1, p_perp),
            (perp2, p_perp),
        ]

        transitions = []
        for next_action, prob in outcomes:
            next_state = (
                state[0] + next_action[0],
                state[1] + next_action[1],
            )
            if not self.is_valid_state(next_state):
                next_state = state
            transitions.append((next_state, prob))

        return transitions

grid_b = WarehouseMDP_Slippery()
V_b, n_b = value_iteration(grid_b)
pi_b = extract_policy(grid_b, V_b)

print("=== EXPERIMENTO B (Piso resbaloso p = 0.40) ===")
print(f"Convergencia VI: {n_b} iteraciones")
print(f"Valor en START (0, 0): {V_b[grid_b.start]:+.3f}\n")

print("Valores V*(s):")
print_values(grid_b, V_b)

print("\nPolítica óptima π*(s):")
print_policy(grid_b, pi_b)

curr = grid_b.start
path_b = [curr]
while not grid_b.is_terminal(curr):
    a = pi_b[curr]
    curr = (curr[0] + a[0], curr[1] + a[1])
    path_b.append(curr)

print("\nTrayectoria desde START:", " → ".join(str(s) for s in path_b))
print("Destino alcanzado:", "ENTREGA +10" if path_b[-1] == (0, 5) else "CARGA +2")

=== EXPERIMENTO B (Piso resbaloso p = 0.40) ===
Convergencia VI: 22 iteraciones
Valor en START (0, 0): -2.706

Valores V*(s):
 -2.706 |  -1.809 |  -0.797 |   WALL   |  +7.607 | +10.000
 -2.652 |   WALL   |  +0.395 |  +2.104 |  +3.670 |  +7.607
 -1.744 |  -0.670 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.831 |  -0.774 |  +0.534 |  -1.137 |  -2.152 | -10.000
 -2.785 |  -3.929 |   WALL   |  -2.152 |  -2.974 |  -4.041

Política óptima π*(s):
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

Trayectoria desde START: (0, 0) → (0, 1) → (0, 2) → (1, 2) → (2, 2)
Destino alcanzado: CARGA +2


**Conclusión del Experimento B:**  
El robot mantiene su decisión de acudir a la **estación de carga +2**, pero el valor en `START` cae de $-2.575$ a $-2.706$. La pérdida de tracción penaliza fuertemente el rendimiento esperado del agente, reforzando la necesidad de buscar una salida inmediata.

---
### Experimento C — Más paciencia (`gamma = 0.99`)

**Predicción antes de ejecutar:**  
Con $\gamma = 0.99$, el horizonte efectivo se amplía notablemente y las recompensas futuras casi no se deprecian ($0.99^7 \approx 0.932$, frente a $0.90^7 \approx 0.478$). El agente se vuelve sumamente "paciente" y la diferencia entre recibir $+10$ y $+2$ cobra mucho más peso que el costo de dar los pasos requeridos para llegar. Por ende, la política óptima desde `START` preferirá la **entrega +10**.

In [9]:
grid_c = WarehouseMDP()
grid_c.gamma = 0.99

V_c, n_c = value_iteration(grid_c)
pi_c = extract_policy(grid_c, V_c)

print("=== EXPERIMENTO C (gamma = 0.99) ===")
print(f"Convergencia VI: {n_c} iteraciones")
print(f"Valor en START (0, 0): {V_c[grid_c.start]:+.3f}\n")

print("Valores V*(s):")
print_values(grid_c, V_c)

print("\nPolítica óptima π*(s):")
print_policy(grid_c, pi_c)

curr = grid_c.start
path_c = [curr]
while not grid_c.is_terminal(curr):
    a = pi_c[curr]
    curr = (curr[0] + a[0], curr[1] + a[1])
    path_c.append(curr)

print("\nTrayectoria desde START:", " → ".join(str(s) for s in path_c))
print("Destino alcanzado:", "ENTREGA +10" if path_c[-1] == (0, 5) else "CARGA +2")

=== EXPERIMENTO C (gamma = 0.99) ===
Convergencia VI: 24 iteraciones
Valor en START (0, 0): -1.456

Valores V*(s):
 -1.456 |  -0.310 |  +0.809 |   WALL   |  +8.601 | +10.000
 -2.179 |   WALL   |  +2.003 |  +4.118 |  +5.354 |  +8.601
 -1.081 |  +0.119 |  +2.000 |  +2.913 |   WALL   |  +7.395
 -1.606 |  -0.467 |  +0.799 |  +0.817 |  -0.359 | -10.000
 -2.752 |  -3.737 |   WALL   |  -0.359 |  -1.408 |  -2.892

Política óptima π*(s):
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  →  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

Trayectoria desde START: (0, 0) → (0, 1) → (0, 2) → (1, 2) → (1, 3) → (1, 4) → (1, 5) → (0, 5)
Destino alcanzado: ENTREGA +10


**Conclusión del Experimento C:**  
Tal como se predijo, con $\gamma = 0.99$ la política valora mucho más la recompensa lejana $+10$. En `(1, 2)` la flecha apunta hacia la derecha $\rightarrow$ y el robot se dirige a **ENTREGA +10**.

---
### Bonus — Umbral crítico de `living_reward`

Determinamos mediante búsqueda numérica el valor aproximado de `living_reward` en el que la política óptima desde `START` conmuta entre ir a la estación de **carga +2** e intentar llegar a la **entrega +10**.

In [10]:
low = -1.0
high = 0.0

def reaches_delivery(living_rew):
    g = WarehouseMDP()
    g.living_reward = living_rew
    V, _ = value_iteration(g, threshold=1e-5)
    pi = extract_policy(g, V)
    s = g.start
    visited = set()
    while not g.is_terminal(s) and s not in visited:
        visited.add(s)
        a = pi[s]
        s = (s[0] + a[0], s[1] + a[1])
    return s == (0, 5)

for _ in range(25):
    mid = (low + high) / 2
    if reaches_delivery(mid):
        high = mid
    else:
        low = mid

umbral = (low + high) / 2
print(f"Umbral crítico de living_reward: {umbral:.4f}")

# Evaluación en torno al umbral
for test_r in [umbral - 0.01, umbral + 0.01]:
    g = WarehouseMDP()
    g.living_reward = test_r
    V, _ = value_iteration(g)
    pi = extract_policy(g, V)
    s = g.start
    while not g.is_terminal(s):
        a = pi[s]
        s = (s[0] + a[0], s[1] + a[1])
    destino = "ENTREGA +10" if s == (0, 5) else "CARGA +2"
    print(f"living_reward = {test_r:+.3f} -> Destino desde START: {destino}")

Umbral crítico de living_reward: -0.7983
living_reward = -0.808 -> Destino desde START: CARGA +2
living_reward = -0.788 -> Destino desde START: ENTREGA +10


**Conclusión del Bonus:**  
El valor umbral a partir del cual cambia la decisión desde `START` es aproximadamente:
$$ \text{living\_reward}^* \approx -0.798 $$

- Para $\text{living\_reward} < -0.798$ (costo por paso más severo, como $-1.0$), el robot prefiere la ruta corta a la **carga +2**.
- Para $\text{living\_reward} \ge -0.798$ (costo por paso más suave, como $-0.5$ o $-0.1$), el costo de las transiciones adicionales compensa el beneficio de alcanzar la **entrega +10**.